# 自动化专业导师模型 - 免费微调

基于 Google Colab 免费 GPU + Qwen2.5-1.5B + LoRA 微调

**步骤：**
1. 运行时 → 更改运行时类型 → 选择 T4 GPU
2. 按顺序运行每个单元格
3. 微调完成后下载模型

## 1. 安装依赖

In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes trl

## 2. 上传训练数据

上传 `automation_advisor.jsonl` 文件（151条训练数据）

In [ ]:
from google.colab import files
import os

# 上传数据文件
print("请上传 automation_advisor.jsonl 文件：")
uploaded = files.upload()

# 确认文件
for fn in uploaded:
    print(f'已上传: {fn} ({len(uploaded[fn])} bytes)')

## 3. 加载和处理数据

In [ ]:
import json
from datasets import Dataset

# 加载数据
data = []
with open("automation_advisor.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

print(f"加载 {len(data)} 条训练数据")
print(f"示例: {data[0]['instruction'][:50]}...")

# 转换为对话格式
def format_to_chat(example):
    messages = []
    if example.get("system"):
        messages.append({"role": "system", "content": example["system"]})
    user_content = example["instruction"]
    if example.get("input"):
        user_content += "\n" + example["input"]
    messages.append({"role": "user", "content": user_content})
    messages.append({"role": "assistant", "content": example["output"]})
    return {"messages": messages}

chat_data = [format_to_chat(d) for d in data]
dataset = Dataset.from_list(chat_data)
print(f"数据集准备完成: {dataset}")

## 4. 加载基座模型

使用 Qwen2.5-1.5B-Instruct（小模型，免费 GPU 够用）

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"正在加载模型: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"模型加载完成！参数量: {model.num_parameters() / 1e9:.2f}B")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU显存: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f}GB")

## 5. 配置 LoRA 并开始训练

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# LoRA 配置
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# 训练配置
training_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=1024,
    report_to="none",
)

# 创建训练器
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("开始训练...")
print(f"训练样本数: {len(dataset)}")
print(f"训练轮数: {training_config.num_train_epochs}")
print(f"学习率: {training_config.learning_rate}")
print(f"LoRA rank: {lora_config.r}")
print("-" * 50)

trainer.train()
print("\n训练完成！")

## 6. 测试微调效果

In [ ]:
def chat(question, system_prompt=None):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# 测试问题
system = "你是一位经验丰富的本科自动化专业指导导师。"

test_questions = [
    "自动化专业大一应该学好哪些课程？",
    "考研还是就业，很纠结怎么办？",
    "嵌入式开发怎么入门？",
]

for q in test_questions:
    print(f"\n{'='*50}")
    print(f"问题: {q}")
    print(f"{'='*50}")
    answer = chat(q, system)
    print(f"回答: {answer}")

## 7. 保存并下载模型

In [ ]:
import shutil

# 保存 LoRA 适配器
save_path = "./automation_advisor_lora"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"模型已保存到: {save_path}")

# 打包下载
shutil.make_archive("automation_advisor_lora", 'zip', save_path)
print("正在下载...")
files.download("automation_advisor_lora.zip")
print("下载完成！")

## 8.（可选）上传到 Hugging Face Hub

In [ ]:
# 取消注释以下代码，上传到 Hugging Face
# 需要先在 https://huggingface.co/settings/tokens 创建 token

# from huggingface_hub import login
# login(token="your_hf_token_here")
# model.push_to_hub("your-username/automation-advisor-lora")
# tokenizer.push_to_hub("your-username/automation-advisor-lora")
# print("上传完成！")